# 04 — Inference Demo

Uses `AdvisoryPipeline` (retrieval-only mode) directly — **runs on CPU, no weights needed**.
Shows the full RAG loop: query → retrieved passages (TF-IDF, `top_k: 5`) → weather context
(mock provider) → answer → safety validation.

> Note: the *Agriculture & Climate SLM Challenge* competition **closed on Sep 9 2026**
> and its **submission format is TBD**, so this project is maintained as a portfolio demo
> rather than a live competition entry.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists() and (ROOT.parent / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.inference.pipeline import AdvisoryPipeline

pipe = AdvisoryPipeline(str(ROOT / "config.yaml"))
print("pipeline ready")

## English query

`language=None` lets the pipeline auto-detect; English queries take the English path.

In [ ]:
result = pipe.answer(
    question="My rice crop is at flowering stage and leaves are turning yellow. What should I do?",
    location="Ludhiana, Punjab",
    crop="rice",
    crop_stage="flowering",
    language=None,
)

print("ANSWER:", result["answer"][:800])
print()
print("WEATHER CONTEXT:", result["weather_context"][:500])
print()
print("DETECTED LANGUAGE:", result["query_meta"].get("language"))
print("SAFETY:", result["safety"])
print()
print("SOURCES:")
for s in result["sources"]:
    print(f" - {s['id']}  (score={s['score']:.3f})  {s['text'][:120]}...")

## Hindi query (Devanagari)

The query classifier uses a Devanagari heuristic to detect Hindi; technical agronomy terms
are kept as-is rather than blindly translated.

In [ ]:
result_hi = pipe.answer(
    question="गेहूं की फसल में पीले पत्ते दिख रहे हैं, क्या करें?",
    location="Indore, Madhya Pradesh",
    crop="wheat",
    crop_stage="vegetative",
    language=None,
)

print("ANSWER:", result_hi["answer"][:800])
print()
print("DETECTED LANGUAGE:", result_hi["query_meta"].get("language"))
print("SAFETY:", result_hi["safety"])
print()
print("SOURCES:")
for s in result_hi["sources"]:
    print(f" - {s['id']}  (score={s['score']:.3f})  {s['text'][:120]}...")

## What the result dict contains

- `answer` — final recommendation string (post safety validation)
- `sources` — retrieved passages as `{"id", "score", "text"}` (RAG evidence)
- `weather_context` — climate text from the weather provider (`mock` in this config)
- `query_meta` — parsed query info (detected language, crop, stage, …)
- `safety` — `{"passed", "flags", "risk_level"}` from the safety validator

The same call pattern powers `app/app.py` (Gradio demo) — see that file for the UI wrapper.